In this notebook I want to compute phase-amplitude coupling

In [ ]:
import os
print(os.getcwd())


In [ ]:
import numpy as np
import mne
from tensorpac import Pac
from mne.stats import permutation_cluster_test, permutation_cluster_1samp_test

import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

import statsmodels.formula.api as smf
import warnings 

# I only want to see warnings once
warnings.filterwarnings('ignore')


In [ ]:
base_dir = '/Volumes/T7 Shield' # this is the root directory for most un-archived data and results
load_dir = f'{base_dir}/work/qasims01/MemoryBanditData/EMU'

# Load the demographics data to get information about who has run the task
demo_file = f'{base_dir}/projects/guLab/Salman/EMU/MemoryBanditsDatabase.xlsx'
demo_data = pd.read_excel(demo_file, engine='openpyxl')
subj_df = demo_data[(demo_data.LongVersion==1) & (demo_data.Minerva==1)]

# Let's get rid of Iowa patients for now 

# subj_df = subj_df[subj_df.Site=='UI']
subj_df = subj_df[~subj_df['MSSMCode'].isin(['MS015', 'MS024', 'MS034', 'MS038'])]


days = ['Day1', 'Day2']
subj_ids = subj_df.MSSMCode.values
subj_formats = subj_df.format.values
subj_sites = subj_df.Site.values
subj_days = subj_df.Day2.values

In [ ]:
# load behavioral data
learn_df = pd.read_csv(f'{load_dir}/learn_df_RWH.csv')
combined_df = pd.read_csv(f'{load_dir}/full_df_RWH.csv')

learn_df['gamble_rt_log'] = np.log(learn_df.gamble_rt)

combined_df['gamble_rt_log'] = np.log(combined_df.gamble_rt_gamble)
combined_df['recog_rt_log'] = np.log(combined_df.recog_rt)

combined_df['memory_outcome'] = combined_df[['hits', 'misses', 'correct_rejections', 
                                       'false_alarms']].idxmax(1).to_frame('memory_outcome') 

In [ ]:
# First, let's drop reaction times that are physiologically too fast
learn_df = learn_df[learn_df.gamble_rt>0.3]
drop_trials = pd.concat([combined_df[combined_df.gamble_rt_gamble<0.3], 
                          combined_df[combined_df.recog_rt<0.3]])
combined_df.drop(index=drop_trials.index, inplace=True)

# Next, let's drop reaction times that are slower than the 97.5th percentile of each subject's reaction times
for subj in learn_df.participant.unique():
    rt_threshold = np.percentile(learn_df[learn_df.participant==subj].gamble_rt, 95)
    learn_df.loc[(learn_df.participant==subj) & (learn_df.gamble_rt>rt_threshold), 'gamble_rt'] = np.nan
    combined_df.loc[(combined_df.participant==subj) & (combined_df.gamble_rt_gamble>rt_threshold), 'gamble_rt_gamble'] = np.nan

learn_df.dropna(subset=['gamble_rt'], inplace=True)
combined_df.dropna(subset=['gamble_rt_gamble'], inplace=True)

# 3/18/2025

# recover the latent Q-values
combined_df['Q'] = combined_df.rpe + combined_df.reward_gamble
# compute the prior trial's rpe:
combined_df['prior_rpe'] = combined_df.groupby('participant')['rpe'].shift(1)
# compute the trials since last reversal
reversal_trials = [16, 33, 48, 65]
combined_df['trials_since_reversal'] = combined_df.apply(lambda x: np.mod(x.trials_gamble, reversal_trials[int(x.block_gamble-2)]) if x.block_gamble>1 else x.trials_gamble, axis=1)
# recode hits as 0.5 and misses as -0.5
combined_df['phit'] = combined_df.apply(lambda x: 0.5 if x.hits==True else -0.5 if x.hits==False else np.nan, axis=1)
# recode reward  as 0.5 and non-reward as -0.5
combined_df['preward'] = combined_df.apply(lambda x: 0.5 if x.reward_gamble==1 else -0.5 if x.reward_gamble==0 else np.nan, axis=1)
# recode choice==1 as 0.5 and choice==0 as -0.5
combined_df['pgamble'] = combined_df.apply(lambda x: 0.5 if x.gamble_gamble==1 else -0.5 if x.gamble_gamble==0 else np.nan, axis=1)

# Gelman standardization 
for subj in combined_df.participant.unique():

        combined_df.loc[combined_df.participant==subj, 
                'zQ'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'Q'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'Q'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'Q']))
        combined_df.loc[combined_df.participant==subj,
                'zprior_rpe'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'prior_rpe'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'prior_rpe'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'prior_rpe']))
        combined_df.loc[combined_df.participant==subj, 
                        'zrpe'] = (combined_df.loc[combined_df.participant==subj, 
                                                                                'rpe'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                        'rpe'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                        'rpe']))
        combined_df.loc[combined_df.participant==subj, 
                'zpm'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'DPRIME'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'DPRIME'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'DPRIME']))

        combined_df.loc[combined_df.participant==subj,
                'ztrials_since_reversal'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'trials_since_reversal'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'trials_since_reversal'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'trials_since_reversal']))
        # compute the reaction-time based EV with res[ect to each subject's min and max RT
        RT_inferred_EV_subj =  1 - ((combined_df.loc[combined_df.participant==subj, 
                                                     'gamble_rt_log'] - np.nanmin(combined_df.loc[combined_df.participant==subj, 
                                                                                                  'gamble_rt_log'])) / (np.nanmax(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                                  'gamble_rt_log']) - np.nanmin(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                                                                                'gamble_rt_log'])))
        # compute 
        combined_df.loc[combined_df.participant==subj,
                'RT_rpe'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'reward_gamble'] - RT_inferred_EV_subj)

        combined_df.loc[combined_df.participant==subj,
                'zRT_rpe'] = (combined_df.loc[combined_df.participant==subj, 
                                                                        'RT_rpe'] - np.nanmean(combined_df.loc[combined_df.participant==subj, 
                                                                                                'RT_rpe'])) / (2*np.nanstd(combined_df.loc[combined_df.participant==subj, 
                                                                                                                                'RT_rpe']))


In [ ]:
# Explicitly define a list of analysis events and the baseline event. Should correspond to the dict

evs = {'gamble_start':[-1.0, 0], 
       'feedback_start': [0, 1.5],
       'baseline_start': [0, 0.75]}

baseline_ev = 'baseline_start'
analysis_evs = ['feedback_start']

day = 'Day1'

for i in analysis_evs: 
    if i not in evs.keys(): 
        raise ValueError('This analysis event is not in the dictionary')

In [ ]:
# # get where phase frequencies start / finish / number
# phase_frequency = 'low'
# f_pha_start, f_pha_end = 4, 9
# f_pha_nb = 20
# f_pha_mid = np.linspace(f_pha_start, f_pha_end, f_pha_nb)
# f_pha = np.c_[f_pha_mid - f_pha_mid / 4., f_pha_mid + f_pha_mid / 4.]
# # get where amplitude frequencies start / finish / number
# amp_frequency = 'hfa'

# f_amp_start, f_amp_end = 70, 200
# f_amp_nb = 10
# f_amp_mid = np.linspace(f_amp_start, f_amp_end, f_amp_nb)
# f_amp = np.c_[f_amp_mid - f_amp_mid / 8., f_amp_mid + f_amp_mid / 8.]

In [ ]:
# get where phase frequencies start / finish / number
phase_frequency = 'low'
f_pha_start, f_pha_end = 13, 30
f_pha_nb = 20
f_pha_mid = np.linspace(f_pha_start, f_pha_end, f_pha_nb)
f_pha = np.c_[f_pha_mid - f_pha_mid / 4., f_pha_mid + f_pha_mid / 4.]
# get where amplitude frequencies start / finish / number
amp_frequency = 'hfa'

f_amp_start, f_amp_end = 70, 200
f_amp_nb = 10
f_amp_mid = np.linspace(f_amp_start, f_amp_end, f_amp_nb)
f_amp = np.c_[f_amp_mid - f_amp_mid / 8., f_amp_mid + f_amp_mid / 8.]

## Compute within-region PAC

In [ ]:
save_dir_plots = f'{base_dir}/work/qasims01/MemoryBanditData/EMU/plots'
save_dir_data = f'{base_dir}/work/qasims01/MemoryBanditData/EMU/dataframes'

n_perm = 200  # number of permutations
# define the Pac object (gaussian-copula PAC) 
p = Pac(idpac=(6, 2, 0), f_pha=f_pha, f_amp=f_amp, dcomplex='wavelet')


In [ ]:
day = 'Day1'

rois = ['HPC', 'OFC', 'dmPFC', 'AMY', 'dlPFC', 'ACC']

for subj_id in subj_ids:
    all_dfs = [] 

    event = 'feedback_start'
    baseline_event = 'baseline_start'
    # baseline_start_mem
    loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
    save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'

    # For each subject, load the epochs of interest and the baseline epoch, as well as the electrode dataframe 
    epochs_reref = mne.read_epochs(f'{save_path}/{event}-epo.fif',  preload=True) 

    elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df')

    buf_ms = 1000 # buffer in ms
    edges = buf_ms * (epochs_reref.info['sfreq']/1000) # convert ms to samples

    # compute true pac and surrogates
    elecs = np.where(elec_df.salman_region.isin(rois))[0] 

    pwise_dfs = []
    if len(elecs)>0:
        for ei in elecs:
            ch_name = elec_df.loc[ei, 'label']
            ch_region = elec_df.loc[ei, 'salman_region']

            real_sig = np.squeeze(epochs_reref.copy().pick(ch_name)._data)

            epoch_pac = p.filterfit(epochs_reref.info['sfreq'], 
                                real_sig, 
                                n_perm=n_perm, n_jobs=-1,
                                edges=edges, random_state=0)


            surr_mean = p.surrogates.mean(axis=0)
            surr_std = p.surrogates.std(axis=0)
            z_scored_pac = (epoch_pac-surr_mean)/surr_std

            # Save out the z_scored pac so I can do individual analysis on it
            np.save(f'{save_dir_data}/{subj_id}_{ch_name}_beta_pac', z_scored_pac)
            
            # such as: t_obs, clusters, cluster_p_values, h0 = permutation_cluster_1samp_test(np.moveaxis(z_scored_pac, -1, 0), 
                                                                    # n_permutations=1000, n_jobs=-1, tail=1)

            # average the z-scores across phae and frequency amplitudes 
            
#             avg_conn = z_scored_pac.mean(axis=(0, 1))
            
#             # alternative: nan out non-significant pac_values, then sum
#             z_scored_pac[np.abs(z_scored_pac)<=1.96] = np.nan
            
#             sum_conn = np.nansum(z_scored_pac, axis=(0, 1))

#             pwise_df = pd.DataFrame(columns=['participant', 'roi', 'label', 'event', 
#                                                 'avg_conn', 'sum_conn', 'hemi'])

#             pwise_df['avg_conn'] = avg_conn
#             pwise_df['sum_conn'] = sum_conn
#             pwise_df['hemi'] = elec_df[elec_df.label==ch_name].hemisphere.values[0]
#             pwise_df['participant'] = subj_id
#             pwise_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
#             pwise_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]

#             pwise_df['region'] = ch_region
#             pwise_df['event'] = event
#             pwise_df['band0'] = phase_frequency
#             pwise_df['band1'] = amp_frequency
#             pwise_df['trials'] = np.arange(1, len(avg_conn)+1)
#             pwise_df['label'] = ch_name


#             pwise_df = pwise_df.merge(learn_df[learn_df.participant==subj_id], on=['participant', 'trials'])

#             pwise_df['zrpe'] = (pwise_df.rpe - np.nanmean(pwise_df.rpe)) / (2*np.nanstd(pwise_df.rpe))
#             pwise_df['zpm'] = (pwise_df.DPRIME - np.nanmean(pwise_df.DPRIME)) / (2*np.nanstd(pwise_df.DPRIME))

# #             # aggregate
#             pwise_dfs.append(pwise_df)
#     else:
#         # this subject does not have electrodes in both regions
#         continue
    
#     if len(pwise_dfs) > 0:
#         all_df_pt = pd.concat(pwise_dfs)
#         all_df_pt.to_csv(f'{save_dir_data}/{subj_id}_pac_{event}_5_4.csv', index=False)


In [ ]:
rtrials = pwise_df.loc[pwise_df.reward==1, 'trials'].values - 1
nrtrials = pwise_df.loc[pwise_df.reward==0, 'trials'].values - 1

In [ ]:
rpac = z_scored_pac[:, :, rtrials]
nrpac = z_scored_pac[:, :, nrtrials]

In [ ]:
t_obs, clusters, cluster_p_values, h0 = permutation_cluster_test([np.moveaxis(rpac, -1, 0),
                                                                    np.moveaxis(nrpac, -1, 0)], 
                                                                    n_permutations=1000, n_jobs=-1, tail=1)

In [ ]:
# t_obs, clusters, cluster_p_values, h0 = permutation_cluster_1samp_test(np.moveaxis(z_scored_pac, -1, 0), 
#                                                         n_permutations=1000, n_jobs=-1, tail=1)

In [ ]:
cluster_p_values

In [ ]:
f = plt.figure(figsize=(11, 5))
plt.subplot(121)
p.comodulogram(z_scored_pac, title=str(p), cmap='Spectral_r')

In [ ]:
save_dir_data = f'{base_dir}/work/qasims01/MemoryBanditData/EMU/dataframes'
pac_df = []
for subj_id in subj_ids:
    all_df_pt = pd.read_csv(f'{save_dir_data}/{subj_id}_pac_feedback_start_beta.csv')
    pac_df.append(all_df_pt)

pac_df = pd.concat(pac_df)


In [ ]:
pac_df.hemi = pac_df.hemi.str.lower()

In [ ]:
pac_df.hemi.unique()

In [ ]:
sns.lmplot(data=pac_df, x='avg_conn_theta', y='avg_conn_beta', hue='region', scatter=False)

In [ ]:
sns.barplot(data=pac_df, x='region', y='avg_conn_theta', errorbar='se')

In [ ]:
smf.mixedlm("avg_conn_theta ~ 1 + C(region, Treatment('AMY'))", 
            groups='participant',
            data=pac_df).fit().summary()


In [ ]:
f = plt.figure(figsize=(11, 5))
plt.subplot(121)
p.comodulogram(z_scored_pac, title=str(p), cmap='Spectral_r')

In [ ]:
# # get the p-values
# mcp = 'fdr'
# pvalues = p.infer_pvalues(p=0.05, mcp=mcp)

# plt.figure(figsize=(8, 6))
# # plot the non-significant pac in gray
# pac_prep_ns = p.pac.mean(-1).copy()
# pac_prep_ns[pvalues < .05] = np.nan
# p.comodulogram(pac_prep_ns, cmap='gray', vmin=np.nanmin(pac_prep_ns),
#                    vmax=np.nanmax(pac_prep_ns), colorbar=False)
# # plot the significant pac in color
# pac_prep_s = p.pac.mean(-1).copy()
# pac_prep_s[pvalues >= .05] = np.nan
# p.comodulogram(pac_prep_s, cmap='Spectral_r', vmin=np.nanmin(pac_prep_s),
#                    vmax=np.nanmax(pac_prep_s), title='')
# plt.gca().invert_yaxis()
# plt.show()

In [ ]:
# p.comodulogram(z_scored_pac, title=str(p), cmap='Spectral_r')
z_scored_pac = (p.pac - p.surrogates.mean(axis=0)) / p.surrogates.std(axis=0)
f = plt.figure(figsize=(11, 5)) 
plt.subplot(121)
p.comodulogram(z_scored_pac, title=str(p), cmap='Spectral_r')


t_obs, clusters, cluster_p_values, h0 = permutation_cluster_1samp_test(np.moveaxis(z_scored_pac, -1, 0),
                                                                       n_permutations=1000, n_jobs=-1)


In [ ]:
cluster_p_values

In [ ]:
f_pha

In [ ]:
# # Now compute the phase preference: 

# # 1. First, for each electrode, identify the frequency range of of maximal coupling for both low- and high-
# # frequency signals 


# from tensorpac import PreferredPhase

# sig_f_pha = [4,8] 
# sig_f_amp = [90,110] 

# p = PreferredPhase(f_pha=[4, 8], f_amp=(70, 200, 10, 1))


# base_dir = '/sc/arion'
# save_dir_plots = '/sc/arion/work/qasims01/MemoryBanditData/EMU/plots'
# save_dir_data = '/sc/arion/work/qasims01/MemoryBanditData/EMU/dataframes'
# # 
# n_perm = 500  # number of permutations
# # define the Pac object (gaussian-copula PAC) 
# # p = Pac(idpac=(6, 2, 0), f_pha=f_pha, f_amp=f_amp, dcomplex='wavelet')

# day = 'Day1'

# for roi in ['OFC']:
#     all_dfs = [] 
#     for subj_id in subj_ids:

#         event = 'feedback_start'
#         baseline_event = 'baseline_start'
#         # baseline_start_mem
#         loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
#         save_path = f'/sc/arion/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'

#         # For each subject, load the epochs of interest and the baseline epoch, as well as the electrode dataframe 
#         epochs_reref = mne.read_epochs(f'{save_path}/{event}-epo.fif',  preload=True) 

#         elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df')

#         buf_ms = 1350 # special case (10/17) - I want to isolate a  specific window in the OFC 
#         edges = buf_ms * (epochs_reref.info['sfreq']/1000)

#         # compute true pac and surrogates
#         elecs = np.where(elec_df.salman_region == roi)[0] 

#         pwise_dfs = []
#         if len(elecs)>0:
#             for ei in elecs:
#                 # ch_name = epochs_reref.ch_names[ei]
#                 ch_name = elec_df.loc[ei, 'label']

#                 real_sig = np.squeeze(epochs_reref.copy().pick(ch_name)._data)

#                 # Extract the phase and the amplitude :
#                 pha = p.filter(epochs_reref.info['sfreq'], 
#                                real_sig, ftype='phase', n_jobs=1)
#                 amp = p.filter(epochs_reref.info['sfreq'], 
#                                real_sig, ftype='amplitude', n_jobs=1)

#                 # Now, compute the PP :
#                 ampbin, pp, vecbin = p.fit(pha, amp, n_bins=72)

In [ ]:
np.where(f_pha[:, 1]<9)

In [ ]:
f_pha

In [ ]:
np.where((f_pha[:, 0]>=7) & (f_pha[:, 0]<=10))

In [ ]:
save_dir_plots = f'{base_dir}/work/qasims01/MemoryBanditData/EMU/plots'
save_dir_data = f'{base_dir}/work/qasims01/MemoryBanditData/EMU/dataframes'
rois = ['HPC', 'OFC', 'dmPFC', 'AMY', 'dlPFC', 'ACC']

# 
n_perm = 500  # number of permutations
# define the Pac object (gaussian-copula PAC) 
p = Pac(idpac=(6, 2, 0), f_pha=f_pha, f_amp=f_amp, dcomplex='wavelet')

day = 'Day1'

all_dfs = [] 
sig_elecs = []

for subj_id in subj_ids:

    event = 'feedback_start'
    baseline_event = 'baseline_start'
    # baseline_start_mem
    loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
    save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'

    # For each subject, load the epochs of interest and the baseline epoch, as well as the electrode dataframe 
    epochs_reref = mne.read_epochs(f'{save_path}/{event}-epo.fif',  preload=True) 

    elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df')
    ch_names = epochs_reref.ch_names
    del epochs_reref

    elecs = np.where(elec_df.salman_region.isin(rois))[0]     
     
    pwise_dfs = []
    if len(elecs)>0:
        for ei in elecs:
            ch_name = elec_df.loc[ei, 'label']
            ch_region = elec_df.loc[ei, 'salman_region']

            z_scored_pac = np.load(f'{save_dir_data}/{subj_id}_{ch_name}_theta_pac.npy')
            
            t_obs, clusters, cluster_p_values, h0 = permutation_cluster_1samp_test(np.moveaxis(z_scored_pac, -1, 0), 
                                                                   n_permutations=1000, n_jobs=-1, tail=1)
            if any(cluster_p_values<=0.05):
                # print('sig electrode')
                # sig_elecs.append(subj_id)
                f = plt.figure(figsize=(11, 5))
                plt.subplot(121)
                p.comodulogram(z_scored_pac, title=str(p), cmap='Spectral_r')
                plt.savefig(f'{save_dir_plots}/{subj_id}_{ch_name}_theta_pac.pdf', dpi=300)
                plt.close(f)
                significant_pac = 1
            else:
                significant_pac = 0

            # Get just the theta pac only
            # z_scored_pac_theta = z_scored_pac[:, np.where(f_pha[:, 1]<9)[0], :]
            z_scored_pac_theta = z_scored_pac

            avg_conn_theta = z_scored_pac_theta.mean(axis=(0, 1))

            # # Get the pac for alpha (greater or equal to 9 and less than 13
            # z_scored_pac_alpha = z_scored_pac[:, np.where((f_pha[:, 0]>=7) & (f_pha[:, 0]<=10))[0], :]
            # avg_conn_alpha = z_scored_pac_alpha.mean(axis=(0, 1))

            # # Get the pac for beta (greater or equal to 13 and less than 30
            # z_scored_pac_beta = z_scored_pac[:, np.where(f_pha[:, 1]>=16)[0], :]
            # avg_conn_beta = z_scored_pac_beta.mean(axis=(0, 1))

            
            # # alternative: nan out non-significant pac_values, then sum
            # z_scored_pac[np.abs(z_scored_pac)<=1.96] = np.nan
            # z_scored_pac_theta[np.abs(z_scored_pac_theta)<=1.96] = np.nan
            # z_scored_pac_alpha[np.abs(z_scored_pac_alpha)<=1.96] = np.nan
            # z_scored_pac_beta[np.abs(z_scored_pac_beta)<=1.96] = np.nan

            # sum_conn_theta = np.nansum(z_scored_pac_theta, axis=(0, 1))
            # sum_conn_alpha = np.nansum(z_scored_pac_alpha, axis=(0, 1))
            # sum_conn_beta = np.nansum(z_scored_pac_beta, axis=(0, 1))

            pwise_df = pd.DataFrame(columns=['participant', 'roi', 'label', 'event', 
                                                 'hemi'])


            pwise_df['avg_conn_theta'] = avg_conn_theta
            # pwise_df['avg_conn_alpha'] = avg_conn_alpha
            # pwise_df['avg_conn_beta'] = avg_conn_beta
            # pwise_df['sum_conn_theta'] = sum_conn_theta
            # pwise_df['sum_conn_alpha'] = sum_conn_alpha
            # pwise_df['sum_conn_beta'] = sum_conn_beta
            # pwise_df['sum_conn'] = sum_conn
            pwise_df['hemi'] = elec_df[elec_df.label==ch_name].hemisphere.values[0]
            pwise_df['participant'] = subj_id
            pwise_df['age'] = subj_df[subj_df.MSSMCode==subj_id].Age.values[0]
            pwise_df['sex'] = subj_df[subj_df.MSSMCode==subj_id].Sex.str.strip().values[0]

            pwise_df['region'] = ch_region
            pwise_df['event'] = event
            pwise_df['band0'] = phase_frequency
            pwise_df['band1'] = amp_frequency
            pwise_df['trials'] = np.arange(1, len(avg_conn_theta)+1)
            pwise_df['label'] = ch_name


            pwise_df = pwise_df.merge(learn_df[learn_df.participant==subj_id], on=['participant', 'trials'])

            pwise_df['zrpe'] = (pwise_df.rpe - np.nanmean(pwise_df.rpe)) / (2*np.nanstd(pwise_df.rpe))
            pwise_df['zpm'] = (pwise_df.DPRIME - np.nanmean(pwise_df.DPRIME)) / (2*np.nanstd(pwise_df.DPRIME))

            # rtrials = pwise_df.loc[pwise_df.reward==1, 'trials'].values - 1
            # nrtrials = pwise_df.loc[pwise_df.reward==0, 'trials'].values - 1
            # rpac = z_scored_pac[:, :, rtrials]
            # nrpac = z_scored_pac[:, :, nrtrials]

            # t_obs, clusters, cluster_p_values, h0 = permutation_cluster_test([np.moveaxis(rpac, -1, 0),
            #                                                         np.moveaxis(nrpac, -1, 0)], 
            #                                                         n_permutations=1000, n_jobs=-1, tail=1)
            
            # if any(cluster_p_values<=0.05):
            #     # print('sig electrode')
            #     # sig_elecs.append(subj_id)
            #     # f = plt.figure(figsize=(11, 5))
            #     # plt.subplot(121)
            #     # p.comodulogram(t_obs, title=str(p), cmap='Spectral_r')
            #     significant_pac = 1
            # else:
            #     significant_pac = 0

            pwise_df['significant_pac'] = significant_pac

# #             # aggregate
            pwise_dfs.append(pwise_df)
    else:
        # this subject does not have electrodes in both regions
        continue
    
    if len(pwise_dfs) > 0:
        all_df_pt = pd.concat(pwise_dfs)
        all_df_pt.to_csv(f'{save_dir_data}/{subj_id}_pac_{event}_theta.csv', index=False)


In [ ]:
# save_dir_plots = f'{base_dir}/work/qasims01/MemoryBanditData/EMU/plots'
# save_dir_data = f'{base_dir}/work/qasims01/MemoryBanditData/EMU/dataframes'
# rois = ['HPC', 'OFC', 'dmPFC', 'AMY', 'dlPFC', 'ACC']

# # 
# n_perm = 500  # number of permutations
# # define the Pac object (gaussian-copula PAC) 
# p = Pac(idpac=(2, 2, 0), f_pha=f_pha, f_amp=f_amp, dcomplex='wavelet')

# day = 'Day1'

# all_dfs = [] 
# sig_elecs = []


# for region in rois:
#     for subj_id in subj_ids:
#         all_pacs = []
#         event = 'feedback_start'
#         baseline_event = 'baseline_start'
#         # baseline_start_mem
#         loadpath = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}'
#         save_path = f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/neural/{day}'

#         # For each subject, load the epochs of interest and the baseline epoch, as well as the electrode dataframe 
#         epochs_reref = mne.read_epochs(f'{save_path}/{event}-epo.fif',  preload=True) 

#         elec_df = pd.read_csv(f'{base_dir}/projects/guLab/Salman/EphysAnalyses/{subj_id}/Day1_reref_elec_df')
#         ch_names = epochs_reref.ch_names
#         del epochs_reref

#         elecs = np.where(elec_df.salman_region==region)[0]     
        
#         if len(elecs)>0:
#             for ei in elecs:
#                 ch_name = elec_df.loc[ei, 'label']
#                 ch_region = elec_df.loc[ei, 'salman_region']

#                 z_scored_pac = np.load(f'{save_dir_data}/{subj_id}_{ch_name}_theta_pac.npy')
#                 # z_scored_pac[z_scored_pac<=1.96] = np.nan
#                 # all_pacs.append(t_obs)
#                 # t_obs, clusters, cluster_p_values, h0 = permutation_cluster_1samp_test(np.moveaxis(z_scored_pac, -1, 0), 
#                 #                                                     n_permutations=1000, n_jobs=-1, tail=1)
#                 all_pacs.append(z_scored_pac)

#         mean_pac = np.nanmean(all_pacs, axis=0)
#         if np.size(mean_pac) == 1:
#             continue

#         f = plt.figure(figsize=(6, 5))
#         plt.subplot(111)
#         p.comodulogram(mean_pac, title=str(p) + f'{region}', cmap='Spectral_r', vmin=0, vmax=0.2)
#         plt.savefig(f'{save_dir_plots}/{subj_id}_{region}_theta_pac.pdf', dpi=300)
#         plt.close()


In [ ]:
np.size(mean_pac)

In [ ]:
mean_pac

In [ ]:
# mean_hpc_pac = np.mean(np.array(all_pacs), axis=0)
# mean_ofc_pac = np.mean(np.array(all_pacs), axis=0)
mean_acc_pac = np.nanmean(np.array(all_pacs), axis=0)
# mean_amy_pac = np.nanmean(np.array(all_pacs), axis=0)
# mean_dmPFC_pac = np.mean(np.array(all_pacs), axis=0)
# mean_dlPFC_pac = np.mean(np.array(all_pacs), axis=0)


In [ ]:
save_dir_data = f'{base_dir}/work/qasims01/MemoryBanditData/EMU/dataframes'
pac_df = []
for subj_id in subj_ids:
    all_df_pt = pd.read_csv(f'{save_dir_data}/{subj_id}_pac_feedback_start_theta.csv')
    pac_df.append(all_df_pt)

pac_df_theta = pd.concat(pac_df)

In [ ]:
save_dir_data = f'{base_dir}/work/qasims01/MemoryBanditData/EMU/dataframes'
pac_df = []
for subj_id in subj_ids:
    all_df_pt = pd.read_csv(f'{save_dir_data}/{subj_id}_pac_feedback_start_beta.csv')
    pac_df.append(all_df_pt)

pac_df_beta = pd.concat(pac_df)

In [ ]:
# Compare the theta and beta pac to eachother in a paired t-test
pac_df_theta['avg_conn_beta'] = pac_df_beta.avg_conn_theta.values
pac_df_theta['avg_conn_theta'] = pac_df_theta.avg_conn_theta.values

# t-test for each region
from scipy.stats import ttest_rel
for region in pac_df_theta.region.unique():
    print(region)
    t, p = ttest_rel(pac_df_theta[pac_df_theta.region==region].avg_conn_theta,
                        pac_df_theta[pac_df_theta.region==region].avg_conn_beta)
    print(t, p)





In [ ]:
# plot pointplot for each region
f = plt.figure(figsize=(6, 5))
sns.pointplot(data=pac_df_theta, 
                x='avg_conn_theta', y='avg_conn_beta', hue='region')


In [ ]:
pac_df['hemi'] = pac_df['hemi'].str.lower()



In [ ]:
condensed_df = pac_df.groupby(['participant', 'region', 'label', 'hemi']).mean(numeric_only=True).reset_index()
condensed_df


In [ ]:
from scipy.stats import binomtest
significant_obs = condensed_df[(condensed_df.significant_pac==1)].region.value_counts()  # Number of significant observations
total_obs = condensed_df.region.value_counts()  # Number of significant observations
percentage_sig = 100 * significant_obs/total_obs 

lci = percentage_sig.copy()
hci = percentage_sig.copy()
p_val = percentage_sig.copy()

for region in ['ACC', 'dlPFC','dmPFC', 'OFC', 'AMY', 'HPC']:
    binom_gridsig = binomtest(significant_obs[region], 
                   total_obs[region], 
                   0.05)
    lb_gs, ub_gs = binom_gridsig.proportion_ci(confidence_level=0.95, method='wilson')

    lci[region] = lb_gs * 100
    hci[region] = ub_gs * 100
    p_val[region] = binom_gridsig.pvalue * 4

percent_df = pd.DataFrame()
percent_df['mean'] = percentage_sig
percent_df['lci'] = lci
percent_df['hci'] = hci
from statsmodels.stats.multitest import multipletests
multipletests(p_val, method='fdr_bh')[1]

percent_df['p-value'] = p_val

percent_df = percent_df.reset_index().rename(columns={'index':'region'})


In [ ]:
percent_df.sort_values(by='mean', ascending=True, inplace=True)

In [ ]:
percent_df

In [ ]:
# Create the bar plot
plt.figure(figsize=(8, 6))
bars = plt.bar(percent_df['region'], percent_df['mean'], 
               yerr=[percent_df['mean'] - percent_df['lci'], percent_df['hci'] - percent_df['mean']],
               capsize=5, color='skyblue', edgecolor='black')

# Add asterisks for significant p-values
for i, (bar, pval) in enumerate(zip(bars, percent_df['p-value'])):
    if pval <= 0.05:
        plt.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 10, '*', 
                 ha='center', va='bottom', fontsize=14, color='red')

plt.hlines(5, -1, 6, 'r')
# Label the axes
plt.xlabel('Region')
plt.ylabel('Mean')
# plt.title('Bar Plot with Confidence Intervals and Significant Markers')
sns.despine()
# Show the plot
plt.tight_layout()
# plt.show()
plt.savefig(f'{save_dir_plots}/theta_pac_significant_regions.pdf', dpi=300)

In [ ]:
smf.mixedlm("avg_conn_theta ~ 1 + hemi + C(region, Treatment('ACC'))",
            groups='participant',
            data=pac_df).fit().summary()

In [ ]:
smf.mixedlm("avg_conn_beta ~ 1 + hemi + C(region, Treatment('ACC'))",
            groups='participant',
            data=pac_df).fit().summary()

In [ ]:
smf.mixedlm("avg_conn_theta ~ 1 + alpha + beta",
            groups='participant',
            data=pac_df[pac_df.region=='AMY']).fit().summary()

In [ ]:
sns.lmplot(data=pac_df, x='beta', y='avg_conn_theta', scatter=False, hue='region')

In [ ]:

f, ax = plt.subplots(1, 1, figsize=(3, 4), dpi=300)

# sns.barplot(data=pac_df, x='region', y='avg_conn_theta', errorbar='se', order=rois)

# make a stripplot to match
# pac_df['region'] = pac_df['region'].astype('str')
# pac_df['participant'] = pac_df['participant'].astype('str')
# pac_df['avg_conn_theta'] = pac_df['avg_conn_theta'].astype('float')

sns.stripplot(data=pac_df[['region', 'participant', 'avg_conn_theta']].groupby(['region', 'participant']).mean().reset_index(),
              y='avg_conn_theta', 
              x='region',
              ax=ax, 
              dodge=True,
              alpha=0.5, legend=None, order=rois)


# plot a horizontal line at the mean value for each roi2
for i, roi in enumerate(rois):
    ax.hlines(pac_df[(pac_df.region==roi)].avg_conn_theta.mean(), i-0.4, i, color='red')

# ax.set(ylim=[-0.5, 0.7])
sns.despine()

In [ ]:
sns.lmplot(data=pac_df, x='reward', y='avg_conn_theta', scatter=False, hue='region')

In [ ]:
z_scored_pac[:, np.where(f_pha[:, 1]<9)[0], :]

In [ ]:
np.where(f_pha[:, 1]<9)[0]

In [ ]:
z_scored_pac.shape

In [ ]:
elec_df[elec_df.label==ch_name].hemisphere.values[0]

In [ ]:
for trial in range(5):
    f = plt.figure(figsize=(11, 5))
    plt.subplot(121)
    plot_pac = z_scored_pac[:, :, trial].copy()
    plot_pac[np.abs(plot_pac) <= 1.96] = np.nan
    p.comodulogram(plot_pac, title=str(p), cmap='Spectral_r')

In [ ]:
f = plt.figure(figsize=(11, 5))
plt.subplot(121)
p.comodulogram(z_scored_pac, title=str(p), cmap='Spectral_r')


t_obs, clusters, cluster_p_values, h0 = permutation_cluster_1samp_test(np.moveaxis(z_scored_pac, -1, 0), 
                                                                       n_permutations=1000, n_jobs=-1, tail=1)
# # split two conditions:
# p.comodulogram(z_scored_pac[:, :, pwise_df.rpe<0], title=str(p), cmap='Spectral_r')
# plt.subplot(122)
# p.comodulogram(z_scored_pac[:, :, pwise_df.rpe>0], title=str(p), cmap='Spectral_r')
# t_obs, clusters, cluster_p_values, h0 = permutation_cluster_test([np.moveaxis(z_scored_pac[:, :, pwise_df.rpe>0], -1, 0),
#                                                                   np.moveaxis(z_scored_pac[:, :, pwise_df.rpe<0], -1, 0)],
#                                                                        n_permutations=500, n_jobs=-1, tail=1)
# # if len(cluster_p_values) > 0:
#     if any(cluster_p_values <= 0.05):
#         f = plt.figure(figsize=(11, 5))
#         plt.subplot(121)
#         p.comodulogram(z_scored_pac, title=str(p), cmap='Spectral_r')


#         # create new stats image with only significant clusters
#         t_obs_plot = np.nan * np.ones_like(t_obs)
#         t_obs_background = t_obs.copy()
#         for c, p_val in zip(clusters, cluster_p_values):
#             if p_val <= 0.05:
#                 t_obs_plot[c] = t_obs[c]
#                 t_obs_background[c] = np.nan

In [ ]:
f = plt.figure(figsize=(11, 5))
plt.subplot(121)
p.comodulogram(z_scored_pac[:, :, pwise_df.rpe>0], title=str(p), cmap='Spectral_r')


# create new stats image with only significant clusters
t_obs_plot = np.nan * np.ones_like(t_obs)
t_obs_background = t_obs.copy()
for c, p_val in zip(clusters, cluster_p_values):
    if p_val <= 0.15:
        t_obs_plot[c] = t_obs[c]
        t_obs_background[c] = np.nan
plt.subplot(122)
p.comodulogram(t_obs_background, cmap='gray', colorbar=False)
p.comodulogram(t_obs_plot, title='Permutation-tested PAC (against z=0)', cmap='viridis', vmin=0, vmax=np.max(t_obs))
plt.gca().invert_yaxis()
plt.ylabel('')

plt.tight_layout()

Cross-regional PAC (need to debug this)